# RhoBench — Google Colab 실행

**내 컴퓨터를 켜 두지 않고**, 구글 서버(Colab)에서 RhoBench를 돌려
**인터넷 주소 하나로 다른 사람에게 보여 주는** 방법입니다.

> 구글 드라이브 자체는 프로그램을 **실행**하지 못합니다(파일 보관소입니다).
> 대신 같은 구글 서비스인 **Colab**이 실제로 파이썬을 돌릴 수 있어서, 여기에
> 프로그램을 올려 실행하고 공개 주소를 만듭니다.

## 사용법

1. 이 파일을 구글 드라이브에 올린 뒤 더블클릭 → **Google Colaboratory**로 열기
2. 위에서부터 셀을 하나씩 실행(▶ 또는 Shift+Enter)
3. 마지막 셀이 출력하는 **https://....trycloudflare.com** 주소와 **비밀번호**를 참석자에게 알려 주기

## 알아 둘 점

| 항목 | 내용 |
|---|---|
| 비용 | 무료 (Colab 무료 등급) |
| 성능 | CPU 2코어 — 정확도 «빠름» 권장. «표준»은 매우 느립니다 |
| 지속 시간 | 브라우저 탭을 닫거나 90분 이상 방치하면 세션이 끊깁니다 |
| 주소 | 세션마다 새로 발급됩니다 (고정 주소 아님) |
| 보안 | 주소를 아는 사람도 **비밀번호 없이는 아무것도 못 봅니다**. 반드시 비밀번호를 정하세요 |
| 데이터 | 세션이 끝나면 계산 결과가 사라집니다 → 4번 셀에서 구글 드라이브에 저장하도록 설정할 수 있습니다 |

## 1. 프로그램 가져오기

아래 둘 중 **하나만** 실행하세요.

In [ ]:
#@title 방법 A — GitHub에서 내려받기 (저장소가 공개인 경우)
REPO = "https://github.com/clsrn0319-ui/DFT-Workbench"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os, shutil
if os.path.isdir("/content/RhoBench"):
    shutil.rmtree("/content/RhoBench")
!git clone --depth 1 -b $BRANCH $REPO /content/RhoBench
%cd /content/RhoBench
print("\n받아온 파일:", sorted(os.listdir("."))[:12])

In [ ]:
#@title 방법 B — 구글 드라이브에 올려 둔 zip 사용 (비공개 저장소일 때)
#@markdown 프로그램 폴더를 zip으로 압축해 내 드라이브에 올린 뒤, 그 경로를 적으세요.
ZIP_PATH = "/content/drive/MyDrive/RhoBench.zip"  #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")

import os, shutil, zipfile
if os.path.isdir("/content/RhoBench"):
    shutil.rmtree("/content/RhoBench")
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("/content/_unzip")

# zip 안에 폴더가 한 겹 더 있어도 server/ 가 있는 위치를 찾아낸다
root = None
for base, dirs, files in os.walk("/content/_unzip"):
    if "server" in dirs and "web" in dirs:
        root = base
        break
assert root, "zip 안에서 server/ 와 web/ 폴더를 찾지 못했습니다."
shutil.move(root, "/content/RhoBench")
%cd /content/RhoBench
print("\n준비된 파일:", sorted(os.listdir("."))[:12])

## 2. 계산 패키지 설치

3~5분 걸립니다. 중간에 뜨는 빨간 경고 메시지는 대부분 무시해도 됩니다.

In [ ]:
!pip install -q pyscf pyscf-dispersion rdkit pyberny fastapi "uvicorn[standard]" "pydantic>=2"

import pyscf, rdkit, fastapi
print("PySCF", pyscf.__version__, "· RDKit", rdkit.__version__, "· 설치 완료")

## 3. 설치 확인 (선택)

실제 DFT 계산 1건을 돌려 봅니다. 30초~1분.

In [ ]:
!python -m pytest tests/ -q
!python -m scripts.smoke_test

## 4. 시연용 결과 미리 계산 (권장)

시연 중에 계산이 끝나기를 기다리지 않도록, 보여줄 결과를 먼저 만들어 둡니다.
5종 × 약 30초 = 3분 내외.

In [ ]:
#@title 시연 데이터 준비
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
#@markdown 체크하면 계산 결과를 내 드라이브(`MyDrive/RhoBench_data`)에 저장해
#@markdown 세션이 끊겨도 다음에 이어서 쓸 수 있습니다.

import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    target = "/content/drive/MyDrive/RhoBench_data"
    os.makedirs(target, exist_ok=True)
    if os.path.islink("data"):
        os.unlink("data")
    elif os.path.isdir("data"):
        import shutil
        for f in os.listdir("data"):
            shutil.copy2(os.path.join("data", f), target)
        shutil.rmtree("data")
    os.symlink(target, "data")
    print("결과 저장 위치:", target)

!python -m scripts.seed_demo --accuracy 빠름

## 5. 서버 켜고 공개 주소 만들기

이 셀을 실행하면 **인터넷 어디서나 접속 가능한 주소**가 출력됩니다.

> 이 셀은 계속 실행 상태로 두어야 합니다. 멈추면 주소도 사라집니다.

In [ ]:
#@title 서버 + 공개 터널 시작
ACCESS_PASSWORD = ""  #@param {type:"string"}
#@markdown 참석자에게 알려 줄 접속 비밀번호. **반드시 입력하세요** —
#@markdown 비워 두면 무작위로 만들어 아래에 출력합니다.

import os, re, secrets, subprocess, time, urllib.request

PORT = 8000
password = ACCESS_PASSWORD.strip() or secrets.token_urlsafe(9)

# --- cloudflared 내려받기 (계정·설치 불필요한 임시 터널) ---
if not os.path.exists("/usr/local/bin/cloudflared"):
    url = ("https://github.com/cloudflare/cloudflared/releases/latest/"
           "download/cloudflared-linux-amd64")
    urllib.request.urlretrieve(url, "/usr/local/bin/cloudflared")
    os.chmod("/usr/local/bin/cloudflared", 0o755)

# --- RhoBench 서버 시작 ---
env = {**os.environ, "RHOBENCH_ACCESS_PASSWORD": password, "RHOBENCH_WORKERS": "2"}
server = subprocess.Popen(
    ["uvicorn", "server.main:app", "--host", "127.0.0.1", "--port", str(PORT)],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise SystemExit("서버가 시작되지 못했습니다:\n" + server.stdout.read())
        time.sleep(1)
else:
    raise SystemExit("서버 시작 시간 초과")

# --- 공개 터널 시작 후 주소 파싱 ---
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None:
            break
        continue
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

bar = "=" * 66
print("\n" + bar)
if public_url:
    print("  RhoBench 접속 주소 (참석자에게 알려 주세요)\n")
    print(f"      {public_url}\n")
else:
    print("  공개 주소를 만들지 못했습니다.")
    print("  아래 대안을 쓰세요 — 이 노트북을 연 본인만 접속됩니다:\n")
    from google.colab.output import eval_js
    print("     ", eval_js(f"google.colab.kernel.proxyPort({PORT})"), "\n")
print(f"  접속 비밀번호:  {password}\n")
print("  ※ 이 셀을 멈추면 주소가 사라집니다. 시연이 끝날 때까지 두세요.")
print(bar)

## 6. 종료 / 결과 내려받기

시연이 끝나면 아래를 실행해 결과를 내 컴퓨터로 저장하세요.
(4번 셀에서 드라이브 저장을 켜 두었다면 이미 드라이브에 남아 있습니다.)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/RhoBench_결과", "zip", "data")
files.download("/content/RhoBench_결과.zip")